## RQ2: How do tail risk characteristics differ between cryptocurrencies and traditional financial markets?

### Imports and Data

In [ ]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
from numpy import exp, mean
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import norm, t, genpareto, laplace, kurtosis, skew, gaussian_kde, ttest_ind
from scipy.stats import t as student_t  # Student-t distribution for simulation
from statsmodels.graphics.gofplots import qqplot
from arch import arch_model
from itertools import groupby
from operator import itemgetter
# Importing pyextreme packages
from pyextremes import get_extremes, get_model, EVA, plot_mean_residual_life, plot_parameter_stability, plot_return_value_stability, plot_threshold_stability, get_extremes, get_return_periods
from pyextremes.plotting import plot_extremes
from pyextremes import plot_parameter_stability
%matplotlib inline

In [ ]:
def load_and_prepare_assets(asset_configs, base_path):
    """
    Load Excel files for multiple assets, set the correct index, and compute log and simple returns.

    Parameters:
        asset_configs (dict): Dictionary specifying for each asset:
            - file: Excel file name
            - date_col: name of the column to use as datetime index
            - price_col: name of the column containing the price to use
        base_path (str): Path prefix to locate all Excel files

    Returns:
        dict: Dictionary of processed pandas DataFrames with log and simple returns
    """
    asset_data = {}

    for asset_name, config in asset_configs.items():
        file_path = base_path + config['file']
        df = pd.read_excel(file_path)

        # Set datetime index
        df[config['date_col']] = pd.to_datetime(df[config['date_col']])
        df.set_index(config['date_col'], inplace=True)
        df.sort_index(inplace=True)

        # Compute returns
        df['Log_Returns'] = np.log(df[config['price_col']] / df[config['price_col']].shift(1))
        df['Simple_Returns'] = df[config['price_col']].pct_change()
        df.dropna(inplace=True)

        asset_data[asset_name] = df

    return asset_data

In [ ]:
def build_evt_dataset(
    assets, 
    return_type='Log_Returns', 
    quantile=0.95, 
    start_date='2010-01-01', 
    end_date='2025-05-09', 
    use_requested_start_date=False,
    align_dates=False
):
    """
    Aligns a dictionary of asset return series to a common time frame (optional), 
    recalculates thresholds, and supports flexible start date logic.

    Parameters:
        assets (dict): e.g. {'BTC': {'Log_Returns': pd.Series, 'Threshold': float}, ...}
        return_type (str): Column name of return series to use (e.g., 'Log_Returns')
        quantile (float): Tail quantile for EVT threshold
        start_date (str): Requested start date (used if use_requested_start_date=True)
        end_date (str): End date for analysis
        use_requested_start_date (bool): 
            - True → try using start_date for each asset (fallback if too early)
            - False → use the latest common start across all assets
        align_dates (bool):
            - True → intersect dates across all assets
            - False → keep assets independently trimmed

    Returns:
        dict: Aligned (or independently trimmed) asset dictionary with thresholds
    """
    aligned_assets = {}
    effective_start_dates = {}

    # Step 0: Establish effective global start date if needed
    if not use_requested_start_date:
        all_starts = [data[return_type].index.min() for data in assets.values()]
        start_date = max(all_starts)
        print(f"Using maximum common start date across all assets: {start_date.strftime('%Y-%m-%d')}")

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # Step 1: Trim individual assets based on their own available range
    for name, data in assets.items():
        series = data[return_type]
        asset_start = series.index.min()

        if use_requested_start_date and start_date < asset_start:
            print(f"WARNING: {name} has no data before {asset_start.strftime('%Y-%m-%d')}. Using that instead.")
            effective_start = asset_start
        else:
            effective_start = max(start_date, asset_start)

        trimmed_returns = series[effective_start:end_date]
        threshold = np.quantile(trimmed_returns, quantile)

        aligned_assets[name] = {
            return_type: trimmed_returns,
            'Threshold': threshold
        }
        effective_start_dates[name] = effective_start

    # Step 2: Optionally align assets to a common date set
    if align_dates:
        date_sets = {name: data[return_type].index for name, data in aligned_assets.items()}
        common_dates = sorted(set.intersection(*[set(dates) for dates in date_sets.values()]))
        print(f"\nCommon date range has {len(common_dates)} timestamps.\n")

        # Step 3: Apply intersection to all assets
        for name, data in aligned_assets.items():
            original_len = len(data[return_type])
            aligned_returns = data[return_type].loc[common_dates]
            aligned_len = len(aligned_returns)
            removed = original_len - aligned_len
            threshold = np.quantile(aligned_returns, quantile)

            aligned_assets[name][return_type] = aligned_returns
            aligned_assets[name]['Threshold'] = threshold

            print(f"{name}: {original_len} → {aligned_len} (removed {removed} rows) | Effective start: {effective_start_dates[name].strftime('%Y-%m-%d')}")
    else:
        # Just print stats, no alignment
        print(f"\nAssets analyzed independently (no date alignment).\n")
        for name, data in aligned_assets.items():
            series_len = len(data[return_type])
            print(f"{name}: {series_len} rows | Effective start: {effective_start_dates[name].strftime('%Y-%m-%d')}")

    return aligned_assets


In [ ]:
asset_configs = {
    'BTC': {'file': 'btcusd_data_20100719_20250516.xlsx', 'date_col': 'Date', 'price_col': 'Price'},
    'BTC_weekend': {'file': 'btcusd_data_20170817_20241231.xlsx', 'date_col': 'Open Time', 'price_col': 'Close'},
    'ETH': {'file': 'ethusd_data_20180209_20250516.xlsx', 'date_col': 'Date', 'price_col': 'Price'},
    'ETH_weekend': {'file': 'ethusd_data_20170817_20241231.xlsx', 'date_col': 'Open Time', 'price_col': 'Close'},
    'S&P500': {'file': 'sp500_data_19480416_20250516.xlsx', 'date_col': 'Date', 'price_col': 'Price'},
    'Gold': {'file': 'xau_1969-01-31_2025-05-09.xlsx', 'date_col': 'Dates', 'price_col': 'Last Price'},
    'EURUSD': {'file': 'eurusd_1975-01-02_2025-05-16.xlsx', 'date_col': 'Dates', 'price_col': 'Price'}
}

base_path = next(
    path for path in [Path.cwd() / 'Data', *(p / 'Data' for p in Path.cwd().parents)]
    if path.exists()
)
base_path = f"{base_path}/"

assets = load_and_prepare_assets(asset_configs, base_path)

In [ ]:
#-- Updating assets dictionary with threshold values for POT
assets = {
    'BTC': {'Log_Returns': assets['BTC']['Log_Returns'], 'Threshold': 0.10},
    'BTC_weekend': {'Log_Returns': assets['BTC_weekend']['Log_Returns'], 'Threshold': 0.10},
    'ETH': {'Log_Returns': assets['ETH']['Log_Returns'], 'Threshold': 0.10},
    'ETH_weekend': {'Log_Returns': assets['ETH_weekend']['Log_Returns'], 'Threshold': 0.10},
    'S&P500': {'Log_Returns': assets['S&P500']['Log_Returns'], 'Threshold': 0.04},
    'Gold': {'Log_Returns': assets['Gold']['Log_Returns'], 'Threshold': 0.025},
    'EURUSD': {'Log_Returns': assets['EURUSD']['Log_Returns'], 'Threshold': 0.015}
}

### Functions

##### Fitting Distributions

In [ ]:
# --- Fit distributions ---
def fit_distributions(returns):
    params_norm = norm.fit(returns)
    params_t = t.fit(returns)
    params_lap = laplace.fit(returns)
    return params_norm, params_t, params_lap

# --- 3a. GPD and GEV Fit for Exceedances (EVT) ---
def fit_gpd(returns, threshold, r, threshold_quantile=0.95, tail="right"):
    if tail == "right":
        model = EVA(data=returns)
        model.get_extremes("POT", threshold=threshold, r=r) # 9% based on the parameter stability plot and r="7D
        model.fit_model(distribution="genpareto")
        params_gpd = model.model._fit_parameters
        return threshold, params_gpd
    else:
        model = EVA(data=returns)
        model.get_extremes("POT", threshold=threshold, r=r, extremes_type="low") # 9% based on the parameter stability plot and r="7D
        model.fit_model(distribution="genpareto")
        params_gpd = model.model._fit_parameters
        return threshold, params_gpd

def fit_gev(returns, block_size, threshold_quantile=0.95):
    model = EVA(data=returns)
    model.get_extremes("BM", block_size=block_size) # we can set block size to "30.44D" to begin, BUT Apa -> 7D
    model.fit_model(distribution="genextreme")
    params_gev = model.model._fit_parameters
    return params_gev

# --- Plot Distribution Fit ---
def plot_fit(returns, params_norm, params_t):
    x = np.linspace(min(returns), max(returns), 1000)
    sns.histplot(returns, stat='density', bins=100, label='Empirical')
    plt.plot(x, norm.pdf(x, *params_norm), label='Normal',  color='red')
    plt.plot(x, t.pdf(x, *params_t), label='t-distribution',  color='#e67e22')
    plt.legend()
    plt.title('Distribution Fit')
    plt.show()

##### Tails

In [ ]:
# --- Fit upside and downside tails ---
def fit_gpd_tail(returns, r='7D', quantile=0.95, tail='right'):
    if tail == 'right':
        threshold = np.quantile(returns, quantile)
        exceedances = returns[returns > threshold] - threshold
    else:
        threshold = np.quantile(returns, 1 - quantile)
        exceedances = -(returns[returns < threshold] - threshold)
    _, gpd_params = fit_gpd(exceedances, threshold, r)
    
    return threshold, gpd_params

#### Summary Statistics

In [ ]:
# --- Summary Statistics ---
def tail_stats(returns):
    df_t, loc_t, scale_t = stats.t.fit(returns)
    return {
        'Skewness': skew(returns),
        'Kurtosis': kurtosis(returns, fisher=False),
        'VaR_95': np.percentile(returns, 5),
        'ES_95': returns[returns < np.percentile(returns, 5)].mean(),
        'VaR_99': np.percentile(returns, 1),
        'ES_99': returns[returns < np.percentile(returns, 1)].mean(),
        'T-test' : perform_t_test(returns, (df_t, loc_t, scale_t))
    }
    
# Helper function for tail_stats()
def print_tail_stats(stats_dict, asset_name="Asset"):
    print(f"\nTail Statistics for {asset_name}:")
    print("-" * (25 + len(asset_name)))
    for key, value in stats_dict.items():
        if key in ['VaR_95', 'VaR_99', 'ES_95', 'ES_99']:
            print(f"{key:<10}: {value * 100:>11.2f}%")
        else:
            print(f"{key:<10}: {value:>12.6f}")

# --- Export summary table ---
def export_summary_table(df, filename='tail_risk_summary.csv'):
    df.to_csv(filename, index=False)
    print(f"Summary table exported to {filename}")

# --- Statistical test for tail shape differences ---
def compare_shape_parameters(returns_a, returns_b, alpha=0.95):
    # Extract shape parameters from both datasets
    _, params_a = fit_gpd_tail(returns_a, quantile=alpha)
    _, params_b = fit_gpd_tail(returns_b, quantile=alpha)
    xi_a = params_a[0]
    xi_b = params_b[0]
    # Bootstrap difference test
    diffs = []
    for _ in range(500):
        sample_a = np.random.choice(returns_a, size=len(returns_a), replace=True)
        sample_b = np.random.choice(returns_b, size=len(returns_b), replace=True)
        try:
            _, xi_a_bs = fit_gpd_tail(sample_a, quantile=alpha)
            _, xi_b_bs = fit_gpd_tail(sample_b, quantile=alpha)
            diffs.append(xi_a_bs[0] - xi_b_bs[0])
        except:
            continue
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    print(f"95% CI for xi difference (A - B): [{ci_low:.4f}, {ci_high:.4f}]")
    return ci_low, ci_high


In [ ]:
# --- Compile Comparison Table and Plots ---
summary = pd.DataFrame(columns=['Asset', 'Skewness', 'Kurtosis', 'GPD_xi', 'VaR_95', 'ES_95'])

#### Center of the Distribution

In [ ]:
# --- T-test interpretation helper ---
def interpret_t_test(t_stat, p_value, alpha=0.05):
    print("\n--- T-test Interpretation ---")
    if p_value < alpha:
        print(f"p-value = {p_value:.4f} < {alpha}: Reject the null hypothesis.")
        print("The mean of returns is significantly different from the mean of the t-distribution sample. We have enough evidence to reject H0")
    else:
        print(f"p-value = {p_value:.4f} ≥ {alpha}: Fail to reject the null hypothesis.")
        print("The t-distribution appears to reasonably match the average return behavior of the asset. We don't have enough evidence to reject H0")

# --- Perform two-sample t-test: return stat + p-value ---
def perform_t_test(empirical_data, dist_params):
    df_t, loc_t, scale_t = dist_params
    simulated_sample = stats.t.rvs(df=df_t, loc=loc_t, scale=scale_t, size=len(empirical_data))
    t_stat, p_val = stats.ttest_ind(empirical_data, simulated_sample, equal_var=False)
    return t_stat, p_val

# --- Summary Statistics ---
def tail_stats(returns):
    df_t, loc_t, scale_t = stats.t.fit(returns)
    t_stat, p_val = perform_t_test(returns, (df_t, loc_t, scale_t))
    
    return {
        'Skewness': skew(returns),
        'Kurtosis': kurtosis(returns, fisher=False),
        'VaR_95': np.percentile(returns, 5),
        'ES_95': returns[returns < np.percentile(returns, 5)].mean(),
        'VaR_99': np.percentile(returns, 1),
        'ES_99': returns[returns < np.percentile(returns, 1)].mean(),
        'T-statistic': t_stat,
        'p-value': p_val
    }

# --- Print Summary Statistics ---
def print_tail_stats(stats_dict, asset_name="Asset"):
    print(f"\nTail Statistics for {asset_name}:")
    print("-" * (25 + len(asset_name)))
    for key, value in stats_dict.items():
        if key in ['VaR_95', 'VaR_99', 'ES_95', 'ES_99']:
            print(f"{key:<12}: {value * 100:>10.2f}%")
        elif key in ['T-statistic', 'p-value']:
            print("-" * (25 + len(asset_name)))
            print(f"{key:<12}: {value:>10.4f}")
        else:
            print(f"{key:<12}: {value:>10.6f}")
    
    # Interpret the t-test at the end
    interpret_t_test(stats_dict['T-statistic'], stats_dict['p-value'])


In [ ]:
# === Interpretation Helper ===
def interpret_t_test(t_stat, p_value, alpha=0.05):
    print("\n--- Interpretation ---")
    if p_value < alpha:
        print(f"p-value = {p_value:.4f} < {alpha}: Reject the null hypothesis.")
        print("The mean of returns is significantly different from the mean of the t-distribution sample.")
    else:
        print(f"p-value = {p_value:.4f} ≥ {alpha}: Fail to reject the null hypothesis.")
        print("The t-distribution appears to reasonably match the average return behavior of the compared asset.")

# === Plot histogram + KDE + t-distribution PDF ===
def plot_return_distribution_vs_t(returns, df_t, loc_t, scale_t):
    x_vals = np.linspace(returns.min(), returns.max(), 1000)
    t_pdf = stats.t.pdf(x_vals, df=df_t, loc=loc_t, scale=scale_t)

    plt.figure(figsize=(12, 6))
    sns.histplot(returns, bins=50, kde=True, stat='density', color='skyblue', label='Empirical BTC Returns')
    plt.plot(x_vals, t_pdf, 'r-', lw=2, label=f"Fitted t-Distribution\n(df={df_t:.2f}, loc={loc_t:.4f}, scale={scale_t:.4f})")
    plt.title("Log Returns vs Fitted t-Distribution")
    plt.xlabel("Log Return")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

# === Perform two-sample t-test and interpret ===
def perform_t_test(empirical_data, dist_params, dist_name='t'):
    df_t, loc_t, scale_t = dist_params
    simulated_sample = stats.t.rvs(df=df_t, loc=loc_t, scale=scale_t, size=len(empirical_data))
    t_stat, p_val = stats.ttest_ind(empirical_data, simulated_sample, equal_var=False)
    print(f"\nTwo-sample t-test:\nt-statistic = {t_stat:.4f}, p-value = {p_val:.4f}")
    interpret_t_test(t_stat, p_val)
    return t_stat, p_val

# === Main Analysis Pipeline ===
def analyze_btc_vs_t_distribution(returns):
    df_t, loc_t, scale_t = stats.t.fit(returns)
    perform_t_test(returns, (df_t, loc_t, scale_t))

### Tail Analysis of Selected Distriubtions

In [ ]:
#-- Generating assets dictionary for independent EVT analysis
assets_2017_095 = build_evt_dataset(
    assets=assets,
    return_type='Log_Returns',
    quantile=0.95,
    start_date='2010-01-01',
    end_date='2025-05-28', # because of Gold
    use_requested_start_date=True,
    align_dates=False
)

In [ ]:
# --- Compare GPD Tail (supports tail direction) ---
def compare_gpd_tail(returns, quantile=95, tail='right'):
    """
    Compare the empirical exceedances of returns to a fitted GPD and simulated Normal distribution,
    supporting analysis of either the right or left tail.

    Parameters:
        returns (pd.Series or np.ndarray): Log returns.
        quantile (float): The quantile threshold for exceedances (e.g., 95).
        tail (str): 'right' for positive tail or 'left' for negative tail analysis.

    Returns:
        dict: Summary statistics and simulation results for tail comparison.
    """
    threshold = np.percentile(returns, quantile)

    if tail == 'right':
        exceedances = returns[returns > threshold] - threshold 
    else:
        exceedances = threshold - returns[returns < threshold]

    # Fit GPD and sample
    shape, loc, scale = genpareto.fit(exceedances)
    simulated_gpd = genpareto.rvs(c=shape, loc=loc, scale=scale, size=len(exceedances))

    # Fit Normal Distribution to the full return series
    mu, std = norm.fit(returns)  # fit normal distribution

    # Simulate from fitted normal, filter by threshold
    normal_simulated_all = norm.rvs(loc=mu, scale=std, size=10000)
    if tail == 'right':
        normal_exceedances = normal_simulated_all[normal_simulated_all > threshold] - threshold
    else:
        normal_exceedances = threshold - normal_simulated_all[normal_simulated_all < threshold]
    normal_exceedances = normal_exceedances[:len(exceedances)]

    if tail == 'right':
        tail_log_returns = returns[returns > threshold]
    else:
        tail_log_returns = returns[returns < threshold]

    return {
        "quantile": quantile,
        "threshold": threshold,

        # Summary statistics
        "Empirical Mean": np.mean(exceedances),
        "GPD Mean": np.mean(simulated_gpd),
        "Normal Mean": np.mean(normal_exceedances),

        # Welch t-tests
        "t-statistic_gpd": ttest_ind(exceedances, simulated_gpd, equal_var=False)[0],
        "p-value_gpd": ttest_ind(exceedances, simulated_gpd, equal_var=False)[1],
        "t-statistic_normal": ttest_ind(exceedances, normal_exceedances, equal_var=False)[0],
        "p-value_normal": ttest_ind(exceedances, normal_exceedances, equal_var=False)[1],

        # Distribution parameters
        "Shape parameter (ξ)": shape,
        "Scale parameter (σ)": scale,
        "normal_params": (mu, std),

        # Raw data
        "exceedances": exceedances,
        "simulated_exceedances": simulated_gpd,
        "normal_exceedances": normal_exceedances,
        "tail_log_returns": tail_log_returns,
        "tail": tail
    }


# --- Plot GPD Comparison (supports tail direction) ---
from scipy.stats import gaussian_kde

def plot_gpd_comparison(stats_dict, save_plot=False, Asset=None):
    """
    Plot the histogram of empirical exceedances and KDE curves for GPD and Normal distribution 
    simulated exceedances, all converted to simple returns.
    
    Parameters:
        stats_dict (dict): Output dictionary from compare_gpd_tail_normal().
        save_plot (bool): If True, saves the plot as a PNG file with dpi=300.
    """
    threshold = stats_dict["threshold"]
    tail = stats_dict.get("tail", "right")

    tail_log_returns = stats_dict["tail_log_returns"]
    exceedances = tail_log_returns
    simulated_exceedances = (
        stats_dict["simulated_exceedances"] + threshold if tail == 'right' 
        else threshold - stats_dict["simulated_exceedances"]
    )
    normal_exceedances = (
        stats_dict["normal_exceedances"] + threshold if tail == 'right' 
        else threshold - stats_dict["normal_exceedances"]
    )

    # Convert to simple returns
    empirical_simple = np.exp(exceedances) - 1
    simulated_simple = np.exp(simulated_exceedances) - 1
    normal_simple = np.exp(normal_exceedances) - 1

    x_min = min(empirical_simple.min(), simulated_simple.min(), normal_simple.min())
    x_max = max(empirical_simple.max(), simulated_simple.max(), normal_simple.max())
    x_vals = np.linspace(x_min, x_max, 1000)

    kde_simulated = gaussian_kde(simulated_simple, bw_method='silverman')
    kde_normal = gaussian_kde(normal_simple, bw_method='silverman')

    plt.figure(figsize=(12, 6))

    # Plot empirical histogram
    plt.hist(empirical_simple, bins=30, density=True, color='skyblue', alpha=0.6, label='Empirical Exceedances')

    # Plot simulated KDEs
    plt.plot(x_vals, kde_simulated(x_vals), lw=2, color='orange', label='GPD Simulated')
    plt.plot(x_vals, kde_normal(x_vals), lw=2, color='green', label='Normal Simulated')

    # Mean markers (Empirical mean in RED)
    plt.axvline(empirical_simple.mean(), color='red', linestyle='--', label='Empirical Mean')
    plt.axvline(simulated_simple.mean(), color='orange', linestyle='--', label='GPD Mean')
    plt.axvline(normal_simple.mean(), color='green', linestyle='--', label='Normal Mean')

    plt.title(f"Empirical vs GPD vs Normal Exceedances\n({tail.capitalize()} Tail > {stats_dict['quantile']} quantile)")
    plt.xlabel("Simple Return (converted from log return exceedances)")
    plt.ylabel("Density")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    
    if save_plot:
        filename = f"GPD_Normal_Tail_{tail}_{stats_dict['quantile']}_{asset}.png"
        plt.savefig(filename, dpi=300)
        print(f"Plot saved as {filename}")

    plt.show()


# === Summary display function ===
def print_gpd_comparison_summary(stats_dict, asset_name="Asset", save_latex=False, latex_filename=None):
    """
    Prints and optionally saves a LaTeX table summarizing the GPD tail comparison for GPD vs Normal,
    with bullet points and custom green for 'Yes' in T-test interpretation.
    """

    tail_log_returns = stats_dict["tail_log_returns"]
    simulated_exceedances = stats_dict["simulated_exceedances"]
    normal_exceedances = stats_dict["normal_exceedances"]
    threshold = stats_dict["threshold"]
    tail = stats_dict["tail"]

    if tail == 'right':
        gpd_tail_log_returns = simulated_exceedances + threshold
        normal_tail_log_returns = normal_exceedances + threshold
    else:
        gpd_tail_log_returns = threshold - simulated_exceedances
        normal_tail_log_returns = threshold - normal_exceedances

    empirical_mean_simple = mean(exp(tail_log_returns) - 1)
    gpd_mean_simple = mean(exp(gpd_tail_log_returns) - 1)
    normal_mean_simple = mean(exp(normal_tail_log_returns) - 1)

    # Deeper green for 'Yes', red for 'No'
    interp_gpd = (r"\textcolor{ForestGreen}{\textbf{Yes}}" if stats_dict['p-value_gpd'] >= 0.05 
                  else r"\textcolor{red}{\textbf{No}}")
    interp_normal = (r"\textcolor{ForestGreen}{\textbf{Yes}}" if stats_dict['p-value_normal'] >= 0.05 
                     else r"\textcolor{red}{\textbf{No}}")
    interp_gpd_msg = ("GPD mean fits empirical mean" if stats_dict['p-value_gpd'] >= 0.05 
                      else "GPD mean differs from empirical mean")
    interp_normal_msg = ("Normal mean fits empirical mean" if stats_dict['p-value_normal'] >= 0.05 
                         else "Normal mean differs from empirical mean")

    latex_str = f"""
    \\begin{{table}}[H]
    \\centering
    \\caption{{GPD Tail Comparison Summary for {asset_name} ({stats_dict['quantile']}\\% tail)}}
    \\begin{{tabular}}{{ll}}
    \\toprule
    \\textbf{{Statistic}} & \\textbf{{Value}} \\\\
    \\midrule
    Threshold ($q_{{{stats_dict['quantile']}}}$)            & {threshold:.6f} \\\\
    Empirical Mean (simple return)  & {empirical_mean_simple:.6f} \\\\
    GPD Mean (simple return)        & {gpd_mean_simple:.6f} \\\\
    Normal Mean (simple return)     & {normal_mean_simple:.6f} \\\\
    Shape parameter ($\\xi$)         & {stats_dict['Shape parameter (ξ)']:.4f} \\\\
    Scale parameter ($\\sigma$)      & {stats_dict['Scale parameter (σ)']:.4f} \\\\
    t-statistic GPD                 & {stats_dict['t-statistic_gpd']:.4f} \\\\
    p-value GPD                     & {stats_dict['p-value_gpd']:.4f} \\\\
    t-statistic Normal              & {stats_dict['t-statistic_normal']:.4f} \\\\
    p-value Normal                  & {stats_dict['p-value_normal']:.4f} \\\\
    GPD exceedances used            & {len(stats_dict['exceedances'])} \\\\
    Normal exceedances used         & {len(stats_dict['normal_exceedances'])} \\\\
    \\midrule
    \\textbf{{T-test fit to \\textit{{empirical mean}}?}} & \\\\
    GPD    & {interp_gpd} ({interp_gpd_msg}) \\\\
    Normal & {interp_normal} ({interp_normal_msg}) \\\\
    \\bottomrule
    \\end{{tabular}}
    \\end{{table}}
    """
    
    print("\nLaTeX Table:\n")
    print(latex_str)

    if save_latex:
        if latex_filename is None:
            latex_filename = f"GPD_Normal_Tail_Summary_{asset_name.replace(' ', '_')}_{tail}_{stats_dict['quantile']}.tex"
        with open(latex_filename, 'w', encoding='utf-8') as f:
            f.write(latex_str)
        print(f"\nLaTeX table saved as: {latex_filename}")

#### Bitcoin

In [ ]:
## BTC - Left Tail
# --- Config
asset = 'BTC'
returns = assets_2017_095[asset]['Log_Returns']
plot_saving = False
latex_saving = False
print(f"--- {asset} ---")
#--------------------------
stats_dict = compare_gpd_tail(returns, quantile=5, tail='left')
plot_gpd_comparison(stats_dict, Asset=asset)

stats_dict = compare_gpd_tail(returns, quantile=1, tail='left')
plot_gpd_comparison(stats_dict, Asset=asset)

#### Ethereum

In [ ]:
## ETH - Left Tail
# --- Config
asset = 'ETH'
returns = assets_2017_095[asset]['Log_Returns']
plot_saving = True
latex_saving = False
print(f"--- {asset} ---")
#--------------------------
stats_dict = compare_gpd_tail(returns, quantile=5, tail='left')
plot_gpd_comparison(stats_dict, Asset=asset)

stats_dict = compare_gpd_tail(returns, quantile=1, tail='left')
plot_gpd_comparison(stats_dict, Asset=asset)

#### Gold

In [ ]:
# --- Config
asset = 'Gold'
returns = assets_2017_095[asset]['Log_Returns']
plot_saving = True
latex_saving = False
print(f"--- {asset} ---")
#--------------------------
stats_dict = compare_gpd_tail(returns, quantile=5, tail='left')
plot_gpd_comparison(stats_dict, Asset=asset)

stats_dict = compare_gpd_tail(returns, quantile=1, tail='left')
plot_gpd_comparison(stats_dict, Asset=asset)

### GARCH + EVT

In [ ]:
def plot_garch_tail_fit(
    returns, asset_name='Asset', threshold=None, quantile=0.95, tail='right',
    rescale_factor=10, window_extension=5, print_shape_param=True,
    min_log_density=1e-6, plot_in_simple_returns=False, zoom_percentile=99.5, save_plot=False
):
    """
    Fit GARCH(1,1) to returns, standardize residuals, and fit GPD/Normal/t to the specified tail.
    Optionally plot fitted distributions and histogram in simple return space, while keeping fitting on residuals.

    Parameters:
        returns (pd.Series): Log return series.
        threshold (float or None): If None, Hill-estimator used; else interpreted as quantile.
        quantile (float): Quantile value used if threshold is not None.
        tail (str): 'right' or 'left'.
        rescale_factor (float): Rescales returns to improve GARCH fit.
        window_extension (float): Horizontal zoom range.
        print_shape_param (bool): Print GPD shape parameter.
        min_log_density (float): (Not used; kept for compatibility).
        plot_in_simple_returns (bool): If True, plot in simple return space.
        zoom_percentile (float): Percentile to zoom upper x-limit for simple return plots.
    """
    # Step 1: Fit GARCH and get standardized residuals
    returns = rescale_factor * returns.dropna()
    model = arch_model(returns, vol='GARCH', p=1, q=1, dist='t')
    result = model.fit(disp="off")
    std_resid = result.std_resid.dropna()
    sigma = result.conditional_volatility
    mu = result.params.get('mu', 0)

    data = std_resid

    # Step 2: Threshold
    if threshold is None:
        print("Estimating threshold using Hill-based simulation...")
        n_candidates, mse_matrix, n_star_per_k, true_tail_indices = simulate_optimal_threshold()
        match_info = match_empirical_tail_index(data, n_star_per_k, true_tail_indices, tail=tail)
        threshold = threshold_from_n_star(data, match_info['n_star'], tail=tail)
    else:
        quantile = threshold
        threshold = data.quantile(quantile if tail == 'right' else 1 - quantile)
        print(f"Using quantile-based threshold: {threshold:.6f} (q={quantile}, tail={tail})")

    # Step 3: Exceedances
    if tail == 'right':
        exceedances = data[data > threshold] - threshold
        tail_data = data[data > threshold]
        x_resid = np.linspace(threshold, data.max(), 1000)
        x_excess = x_resid - threshold
    elif tail == 'left':
        exceedances = threshold - data[data < threshold]
        tail_data = data[data < threshold]
        x_resid = np.linspace(data.min(), threshold, 1000)
        x_excess = threshold - x_resid
    else:
        raise ValueError("tail must be 'right' or 'left'")

    print(f"Number of exceedances used for GPD fit: {len(exceedances)}")

    # Step 4: GPD/Normal/t fit
    params_gpd = genpareto.fit(exceedances)
    params_norm = norm.fit(data)
    params_t = t.fit(data)

    if print_shape_param:
        print(f"GPD Shape Parameter (ξ): {params_gpd[0]:.4f}")
        print(f"Interpretation of ξ: {interpret_shape_param(params_gpd[0])}")

    # Step 5: Optional conversion to simple return space
    if plot_in_simple_returns:
        print("Plotting in simple return space (converted after GPD fit)...")
        sigma_scalar = sigma.mean()

        log_returns_tail = mu + sigma_scalar * tail_data
        simple_returns_tail = np.exp(log_returns_tail) - 1

        x_plot = np.exp(mu + sigma_scalar * x_resid) - 1
        x_excess_plot = x_plot - x_plot[0]
        fit_gpd_plot = genpareto.pdf(x_excess_plot, *params_gpd)
        fit_norm_plot = norm.pdf(np.log(1 + x_plot), *params_norm) * 1 / (1 + x_plot)
        fit_t_plot = t.pdf(np.log(1 + x_plot), *params_t) * 1 / (1 + x_plot)

        data_for_hist = simple_returns_tail
        threshold_plot = x_plot[0]
        x_label = 'Simple Return'
    else:
        x_plot = x_resid
        fit_gpd_plot = genpareto.pdf(x_excess, *params_gpd)
        fit_norm_plot = norm.pdf(x_resid, *params_norm)
        fit_t_plot = t.pdf(x_resid, *params_t)
        data_for_hist = tail_data
        threshold_plot = threshold
        x_label = 'Standardized Residual'

    # Step 6: Plot
    fig, ax = plt.subplots(figsize=(10, 6))

    if len(data_for_hist) < 10:
        print(f"Only {len(data_for_hist)} points in tail — histogram may be unreliable.")

    ax.hist(data_for_hist, bins='auto', density=True, alpha=0.4,
            color='skyblue', edgecolor='gray', label='Empirical (Tail)')
    ax.plot(x_plot, fit_gpd_plot, label='GPD Fit (tail)', color='black', linestyle='--')
    ax.plot(x_plot, fit_norm_plot, label='Normal Fit', color='red')
    ax.plot(x_plot, fit_t_plot, label='t-Distribution Fit', color='orange')
    ax.axvline(threshold_plot, color='gray', linestyle=':', label='Threshold')

    # Zoom logic
    if plot_in_simple_returns:
        x_upper = np.percentile(data_for_hist, zoom_percentile)
        if tail == 'right':
            ax.set_xlim(threshold_plot, x_upper)
        elif tail == 'left':
            x_lower = np.percentile(data_for_hist, 100 - zoom_percentile)
            ax.set_xlim(x_lower, threshold_plot)
    else:
        if tail == 'right':
            ax.set_xlim(threshold_plot, x_plot.max())
        elif tail == 'left':
            ax.set_xlim(x_plot.min(), threshold_plot)

    ax.set_title(f"{asset_name} Tail Fit: GPD vs Normal vs t ({'Simple Returns' if plot_in_simple_returns else 'Standardized Residuals'}) — {tail.title()} Tail, {thresh}")
    ax.set_xlabel(x_label)
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    if save_plot == True:
        plt.savefig(f'{asset_name}_GARCH_tail_fit_gpd_normal_t_{tail}_tail_{thresh}.png', dpi=300, bbox_inches='tight')
        print("GARCH + EVT plot successfully saved.")
    plt.show()

In [ ]:
def interpret_shape_param(xi):
    if xi < -1:
        return "Extremely short tail (hard bound)"
    elif xi < -0.5:
        return "Very short tail (strongly bounded)"
    elif xi < -0.1:
        return "Short tail (bounded)"
    elif xi < 0:
        return "Mildly bounded tail (near-exponential)"
    elif xi < 0.1:
        return "Light tail (near exponential)"
    elif xi < 0.3:
        return "Moderate heavy tail"
    elif xi < 0.5:
        return "Heavy tail (finite variance)"
    elif xi < 1.0:
        return "Very heavy tail (infinite variance)"
    else:
        return "Extremely heavy tail (infinite mean)"

#### Bitcoin

In [ ]:
# --- Config
asset = 'BTC'
returns_to_use = assets_2017_095[asset]['Log_Returns']*10 
thresh = 0.95
_tail = 'left'
plot = True
plot_simple_returns = False
print(f"--- {asset} ---")
#--------------------------
plot_garch_tail_fit(
    returns=returns_to_use,
    asset_name=asset,
    tail=_tail,
    threshold=thresh, # if we fill-in the threshold it gets interpreted as the quantile, so it works correctly
    plot_in_simple_returns=plot_simple_returns,
    zoom_percentile=99.5,  # tighter zoom
    save_plot=False
)

In [ ]:
# --- Config
asset = 'BTC'
returns_to_use = assets_2017_095[asset]['Log_Returns']*10 
thresh = 0.99
_tail = 'left'
plot = True
plot_simple_returns = False
print(f"--- {asset} ---")
#--------------------------
plot_garch_tail_fit(
    returns=returns_to_use,
    asset_name=asset,
    tail=_tail,
    threshold=thresh, # if we fill-in the threshold it gets interpreted as the quantile, so it works correctly
    plot_in_simple_returns=plot_simple_returns,
    zoom_percentile=99.5,  # tighter zoom
    save_plot=False
)

#### Ethereum

In [ ]:
# --- Config
asset = 'ETH'
returns_to_use = assets_2017_095[asset]['Log_Returns']*10 
thresh = 0.95
_tail = 'left'
plot = True
plot_simple_returns = False
print(f"--- {asset} ---")
#--------------------------
plot_garch_tail_fit(
    returns=returns_to_use,
    asset_name=asset,
    tail=_tail,
    threshold=thresh, # if we fill-in the threshold it gets interpreted as the quantile, so it works correctly
    plot_in_simple_returns=plot_simple_returns,
    zoom_percentile=99.5,  # tighter zoom
    save_plot=False
)

In [ ]:
# --- Config
asset = 'ETH'
returns_to_use = assets_2017_095[asset]['Log_Returns']*10 
thresh = 0.99
_tail = 'left'
plot = True
plot_simple_returns = False
print(f"--- {asset} ---")
#--------------------------
plot_garch_tail_fit(
    returns=returns_to_use,
    asset_name=asset,
    tail=_tail,
    threshold=thresh, # if we fill-in the threshold it gets interpreted as the quantile, so it works correctly
    plot_in_simple_returns=plot_simple_returns,
    zoom_percentile=99.5,  # tighter zoom
    save_plot=False
)

#### Gold

In [ ]:
# --- Config
asset = 'Gold'
returns_to_use = assets_2017_095[asset]['Log_Returns']*10 
thresh = 0.95
_tail = 'left'
plot = True
plot_simple_returns = False
print(f"--- {asset} ---")
#--------------------------
plot_garch_tail_fit(
    returns=returns_to_use,
    asset_name=asset,
    tail=_tail,
    threshold=thresh, # if we fill-in the threshold it gets interpreted as the quantile, so it works correctly
    plot_in_simple_returns=plot_simple_returns,
    zoom_percentile=99.5,  # tighter zoom
    save_plot=False
)

In [ ]:
# --- Config
asset = 'Gold'
returns_to_use = assets_2017_095[asset]['Log_Returns']*10 
thresh = 0.99
_tail = 'left'
plot = True
plot_simple_returns = False
print(f"--- {asset} ---")
#--------------------------
plot_garch_tail_fit(
    returns=returns_to_use,
    asset_name=asset,
    tail=_tail,
    threshold=thresh, # if we fill-in the threshold it gets interpreted as the quantile, so it works correctly
    plot_in_simple_returns=plot_simple_returns,
    zoom_percentile=99.5,  # tighter zoom
    save_plot=False
)

**Overall:**
* This notebook compares tail risk across cryptocurrencies and traditional assets using log returns.

* The analysis uses **EVT/POT** methods, fitting Generalized Pareto Distributions to extreme returns and comparing these tails against Normal and Student-t style assumptions.

* For Bitcoin’s left tail, the GPD fit performs much better than the Normal distribution: the Normal model significantly understates/extorts the extreme positive-return behavior, while the GPD better matches the empirical exceedances.

* The downside-tail analysis shows meaningful tail risk in BTC, ETH, and Gold. After GARCH filtering, BTC’s left tail appears light-to-moderately heavy, ETH shows a heavy 95% left tail but weaker 99% result due to fewer exceedances, and Gold shows especially heavy downside tail behavior at the 99% level.

* All in all, the notebook suggests that extreme market moves are not well captured by Normal assumptions, and **EVT/GARCH-EVT can be more suitable** for studying crash risk and large tail events, especially when comparing crypto assets with traditional markets.